# AgriScan re-implementation: Kaggle runner

Settings to check first (right sidebar): **Accelerator = GPU T4 x2 or P100**, **Internet = On** (needs a phone-verified account).

The notebook only clones the repo and calls its scripts. All logic lives in the repo, so results are reproducible.
Never paste tokens or passwords into this notebook.

In [ ]:
!nvidia-smi

In [ ]:
REPO_URL = "https://github.com/kabilGa/agriscan-reimpl"

import os
os.chdir("/kaggle/working")
if not os.path.isdir("agriscan-reimpl"):
    !git clone {REPO_URL}
os.chdir("agriscan-reimpl")
!git pull --ff-only && git log --oneline -1

In [ ]:
# Kaggle already ships torch/torchvision; this only adds what is missing.
!pip install -q -r requirements.txt pytest

In [ ]:
# Environment check: the JSON should name a GPU on a GPU session.
!python scripts/smoke_test.py
!cat artifacts/smoke_test.json
!python -m pytest -q tests

## Step 1: data, leakage, clean split

First: **Add Input** (right sidebar) -> search *New Plant Diseases Dataset* (owner `vipoooool`) -> Add. Check its license on the dataset page.
The scripts search for the train/valid/test folders, so the exact nesting does not matter.

In [ ]:
DATA_ROOT = "/kaggle/input/new-plant-diseases-dataset"   # check this path in the right sidebar
!python scripts/01_inspect_data.py --root "{DATA_ROOT}"

In [ ]:
# ~88K images through MobileNetV2: a few minutes on GPU. Features are cached in artifacts/leakage/.
!python scripts/02_leakage.py --root "{DATA_ROOT}"

In [ ]:
# Look at the galleries: which similarity band starts to look like 'the same photo, augmented'?
import glob
from IPython.display import Image, display
for p in sorted(glob.glob("artifacts/leakage/pairs_*.png")):
    print(p)
    display(Image(p))

In [ ]:
# Run once per candidate threshold, AFTER looking at the galleries and tell Claude what you saw.
THRESHOLD = 0.95   # provisional: replace by your choice
!python scripts/03_clean_split.py --threshold {THRESHOLD}

Download `results/*.json`, `splits/*.csv.gz` and the PNG galleries from the Output tab (or `Save Version`), then copy the JSON/CSV into your local repo. Do not put a GitHub token on Kaggle.